# 06_Stable_Diffusion_v1_4 — 셀별 코드 해설 노트북 (Python 초보용)

이 노트북은 원본 `06_Stable_Diffusion_v1_4.ipynb`와 **코드가 완전히 같습니다.** `# 1`, `# 2` … 번호가 붙은 코드 셀 **바로 위에** "[셀 # N] 코드 설명" 칸이 하나씩 추가되었습니다. 설명은 *한 줄 요약 → 코드 흐름 → Python 초보 포인트* 순서입니다.

**Stable Diffusion을 한 문장으로:** 글(프롬프트)을 입력하면 그 내용에 맞는 그림을 만들어 주는 모델입니다. 완전한 잡음에서 출발해, 글의 내용을 참고하며 잡음을 조금씩 걷어 내는 과정을 수십 번 반복해 그림을 완성합니다.

**이 노트북의 특징:** 모델을 만들거나 학습시키지 않습니다. 완성된 모델을 내려받아 **옵션 값을 바꿔 가며** 결과가 어떻게 달라지는지 관찰합니다. 그래서 코드는 짧고, 대부분 "값을 바꿔 여러 장 생성 → 나란히 그리기"의 반복입니다.

**실험하는 옵션 한눈에 보기**

| 옵션 | 뜻 | 실험하는 셀 |
|---|---|---|
| `prompt` | 그리고 싶은 내용(영어 문장) | 4, 5, 11 |
| `guidance_scale` | 프롬프트를 얼마나 **강하게 따를지** | 6, 7 |
| `generator`(seed) | 출발점이 되는 무작위 잡음의 번호 | 8 |
| `negative_prompt` | 그림에 **나오지 않았으면** 하는 것 | 9 |
| `num_inference_steps` | 잡음을 걷어 내는 반복 횟수 | 10 |
| `width`, `height` | 그림의 가로·세로 크기 | 12 |

**준비물:** 이 노트북은 **NVIDIA GPU가 필수**입니다(3번 셀에서 `.to("cuda")`를 사용). 그림 한 장을 만드는 데 GPU로 몇 초~수십 초가 걸립니다.

---

# **Stable Diffusion v1.4 — Text-to-Image Generation**

**Huggingface**의 `diffuser` 라이브러리를 사용하여 **Stable Diffusion v1.4** 모델로 다양한 이미지를 직접 생성해 봅니다.

## 학습 목표
- Stable Diffusion 추론 파이프라인의 구조적 이해: 사전 학습된 모델을 로드하고, 텍스트 프롬프트가 이미지 텐서로 변환되는 전체 과정을 파악한다.
- 생성 파라미터 최적화를 통한 결과 제어
    - 핵심 하이퍼파라미터 변화가 이미지의 품질과 프롬프트 준수율에 미치는 영향을 실험하고 최적의 조합을 도출한다.

---

### 구성 개요

1. Stable Diffusion 모델 로드
2. 텍스트 프롬프트로 이미지 생성
3. 창의성
4. Seed
5. Negative Prompt
6. Inference step
7. Style variation
8. 해상도와 구도

---
## 0) Setup (환경 준비)
- **목적:** 실습에 필요한 패키지 설치

### [셀 # 1] 코드 설명

**한 줄 요약:** 실습에 필요한 라이브러리(`diffusers`, `transformers`, `accelerate`)가 설치되어 있는지 확인하고, 없으면 자동으로 설치합니다.

**코드 흐름**

1. `import importlib.util`, `subprocess`, `sys` — Python 기본 도구 3개입니다. 각각 "패키지 찾아보기", "다른 프로그램(pip) 실행", "현재 Python 정보" 용도입니다.
2. `def ensure_package(pkg_name, import_name=None)` — "확인 후 없으면 설치"를 함수로 묶었습니다.
   - `name = import_name or pkg_name` : `import_name`이 비어 있으면 `pkg_name`을 씁니다.
   - `importlib.util.find_spec(name) is None` : 패키지를 못 찾으면 `None` → 설치가 필요하다는 뜻입니다.
   - `subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])` : 터미널에서 `python -m pip install -q 패키지명`을 실행한 것과 같습니다. `-q`는 설치 과정 출력을 줄이는 옵션입니다.
3. `packages = [...]` — (설치 이름, import 이름) 쌍의 목록입니다.
4. `for pkg, imp in packages:` — 목록을 돌며 함수를 호출합니다.

**각 패키지의 용도**

- `diffusers` : Stable Diffusion 같은 그림 생성 모델을 쉽게 쓰게 해 주는 Hugging Face 라이브러리
- `transformers` : 글(프롬프트)을 이해하는 부분(텍스트 인코더)에 필요
- `accelerate` : 모델을 빠르고 효율적으로 불러오게 도와주는 도구

**Python 초보 포인트**

- `def 함수이름(인자):` 로 함수를 만들고, `함수이름(값)` 으로 실행합니다.
- `try: ... except ... as e:` 는 에러가 나도 프로그램을 멈추지 않고 대신 처리하는 구문입니다.
- `for pkg, imp in packages:` 는 튜플 `("diffusers", "diffusers")`를 두 변수에 나눠 담는 언패킹입니다.
- 새로 설치했다면 커널을 한 번 재시작해야 인식되는 경우가 있습니다.

In [ ]:
# 1
import importlib.util
import subprocess
import sys

def ensure_package(pkg_name: str, import_name=None):
    """
    패키지 설치 여부를 확인하고, 설치되어 있지 않으면 설치합니다.
    """
    name = import_name or pkg_name
    # 패키지가 설치되어 있는지 확인
    if importlib.util.find_spec(name) is None:
        print(f"[install] {pkg_name} 라이브러리를 설치 중입니다... (import name: {name})")
        try:
            # -q 옵션을 추가하여 설치 과정을 간결하게 유지할 수 있습니다.
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])
            print(f"[success] {pkg_name} 설치 완료.")
        except subprocess.CalledProcessError as e:
            print(f"[error] {pkg_name} 설치 실패: {e}")
    else:
        print(f"[ok] {pkg_name} 이미 설치되어 있습니다.")

# 설치가 필요한 패키지 리스트 (패키지명, 임포트명)
# 임포트명이 패키지명과 다른 경우 튜플로 지정합니다.
packages = [
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate")
]

# 루프를 돌며 확인 및 설치
for pkg, imp in packages:
    ensure_package(pkg, imp)


---
## 0-1) 환경 세팅
- **목적:** 실습에 필요한 Hugging Face Access Token을 발급받고 로그인 합니다.
- **관찰 포인트**
    - Hugging Face Access Token 발급
       - Hugging Face 우상단 프로필 아이콘 → Settings → Access Tokens.
       - New token 클릭
       - 이름: 원하는 이름 입력
       - Role: Read (기본값이면 OK)
       - 생성하면 hf_~~~로 시작하는 긴 문자열이 나오는데, 이걸 어딘가에 복사해 두세요.
    - Huggingface 계정 로그인 (access token 필요)

### [셀 # 2] 코드 설명

**한 줄 요약:** 이 셀은 **모든 줄이 주석(`#`)** 이라서 실행해도 아무 일도 일어나지 않습니다. Hugging Face 로그인이 필요한 경우에만 주석을 풀어 사용하는 예비 코드입니다.

**주석을 풀면 하는 일**

- `from huggingface_hub import notebook_login` + `notebook_login()` : 노트북 안에 로그인 창을 띄웁니다. 여기에 Hugging Face 사이트에서 발급받은 Access Token을 붙여 넣습니다.
- `access_token = "hf_..."` : 토큰을 변수에 직접 적어 두는 다른 방법입니다. 3번 셀의 `use_auth_token=access_token`(이것도 주석 처리되어 있음)과 짝을 이룹니다.

**언제 필요한가?** 모델을 내려받을 때 "권한이 없다"거나 "로그인이 필요하다"는 에러가 날 때입니다. 3번 셀이 문제없이 실행되면 이 셀은 그대로 두면 됩니다.

**Python 초보 포인트**

- 줄 맨 앞의 `#` 는 "이 줄은 실행하지 말라"는 표시입니다. 코드를 지우지 않고 잠시 꺼 둘 때 씁니다.
- 주석을 풀려면 맨 앞의 `# `(샵과 공백)을 지웁니다. 주피터에서는 줄을 선택하고 `Ctrl + /` 를 누르면 주석이 켜지고 꺼집니다.

**알아둘 점:** 토큰은 비밀번호와 같습니다. 코드에 직접 적어 둔 채로 노트북 파일을 다른 사람에게 공유하거나 인터넷에 올리지 않도록 주의하세요. 가능하면 `notebook_login()` 방식을 쓰는 것이 안전합니다.

In [ ]:
# 2
# from huggingface_hub import notebook_login

# notebook_login()
# access_token = "hf_여기에_본인의_토큰_입력"

---
## 1) Stable Diffusion 모델 로드
- **목적:** Stable Diffusion 1.4 모델을 로드합니다
- **관찰 포인트**
    - model_id는 우리가 사용할 모델의 ID이며
    - pipe는 StableDiffusionPipeline을 통해 모델을 GPU에 로드

### [셀 # 3] 코드 설명

**한 줄 요약:** Stable Diffusion v1.4 모델을 내려받아 GPU에 올리고, 이 모델이 어떤 부품들로 이루어져 있는지 이름을 출력합니다.

**코드 흐름**

1. `from diffusers import StableDiffusionPipeline` — "글 → 그림" 전체 과정을 하나로 묶은 도구(파이프라인)를 가져옵니다.
2. `model_id = "CompVis/stable-diffusion-v1-4"` — Hugging Face에 올라와 있는 모델의 이름(주소)입니다.
3. `pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")`
   - `from_pretrained(model_id, ...)` : 학습이 끝난 모델을 내려받아 불러옵니다.
   - `torch_dtype=torch.float16` : 숫자를 절반 크기(16비트)로 저장합니다. GPU 메모리를 절반 가까이 아끼고 속도도 빨라집니다.
   - `.to("cuda")` : 모델을 GPU로 옮깁니다.
4. `for name, component in pipe.components.items():` — 파이프라인 안의 부품을 하나씩 꺼내 "이름: 종류"를 출력합니다.

**출력되는 주요 부품**

| 이름 | 역할 |
|---|---|
| `tokenizer` | 문장을 숫자 토큰으로 쪼갭니다. |
| `text_encoder` | 토큰을 "의미를 담은 벡터"로 바꿉니다(CLIP). |
| `unet` | 잡음을 조금씩 걷어 내는 핵심 신경망(04, 05번 노트북의 U-Net과 같은 계열) |
| `scheduler` | 단계마다 잡음을 얼마나 걷어 낼지 정하는 일정표 |
| `vae` | 압축된 작은 표현을 실제 512×512 그림으로 펼쳐 줍니다. |
| `safety_checker`, `feature_extractor` | 부적절한 이미지인지 검사합니다. |

**Python 초보 포인트**

- `pipe.components` 는 딕셔너리이고, `.items()` 는 (이름, 값) 쌍을 하나씩 꺼내 줍니다.
- `type(component).__name__` : 그 객체의 종류(클래스) 이름을 문자열로 얻습니다.
- 괄호 안에서는 줄을 자유롭게 바꿀 수 있습니다. `from_pretrained(` 뒤로 여러 줄에 걸쳐 쓴 것도 한 문장입니다.
- `# use_auth_token=access_token,` 은 2번 셀과 짝을 이루는 주석 처리된 옵션입니다.

**알아둘 점**

- 처음 실행하면 **수 GB**의 파일을 내려받으므로 시간이 걸립니다. 한 번 받으면 컴퓨터에 저장되어 다음부터는 빠릅니다.
- GPU가 없으면 `.to("cuda")`에서 에러가 납니다.
- 이후 모든 셀이 여기서 만든 `pipe`를 사용합니다. 커널을 재시작했다면 이 셀부터 다시 실행해야 합니다.

In [ ]:
# 3
from diffusers import StableDiffusionPipeline
import torch
from IPython.display import display

# 모델 로드 (Stable Diffusion v1-4)
model_id = "CompVis/stable-diffusion-v1-4"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    # use_auth_token=access_token, # 토큰 직접 전달
    torch_dtype=torch.float16   # 메모리 절약을 위해 권장
).to("cuda")

# 파이프라인 내부 구성 요소 출력
for name, component in pipe.components.items():
    print(f"{name}: {type(component).__name__}")

---
## 1-1) 파이프라인 구성 요소와 파라미터 매핑

<table style="margin-left: 0; margin-right: auto;">
    <thead>
        <tr>
            <th style="text-align: left;">파라미터명</th>
            <th style="text-align: left;">연관 내부 객체</th>
            <th style="text-align: left;">역할 설명</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td><b>prompt</b></td>
            <td>tokenizer, text_encoder</td>
            <td>입력한 문장을 수치화하여 이미지의 <b>주제와 방향</b>을 결정</td>
        </tr>
        <tr>
            <td><b>negative_prompt</b></td>
            <td>text_encoder</td>
            <td>이미지에서 <b>제외하고 싶은 요소</b>를 벡터 공간에서 멀어지게 설정</td>
        </tr>
        <tr>
            <td><b>guidance_scale</b></td>
            <td>unet</td>
            <td>AI가 프롬프트를 얼마나 <b>강하게(고집스럽게)</b> 따를지 결정</td>
        </tr>
        <tr>
            <td><b>num_inference_steps</b></td>
            <td>scheduler</td>
            <td>노이즈를 제거하는 <b>반복 횟수</b>. 많을수록 정교하지만 시간이 오래 걸림</td>
        </tr>
        <tr>
            <td><b>generator (seed)</b></td>
            <td>Latent Noise</td>
            <td>이미지의 기초가 되는 <b>난수(노이즈) 패턴</b>을 고정 (동일 구도 재현)</td>
        </tr>
        <tr>
            <td><b>width / height</b></td>
            <td>unet, vae</td>
            <td>출력 이미지의 <b>해상도</b>. VRAM 사용량과 구도(가로/세로)에 영향</td>
        </tr>
    </tbody>
</table>

---
## 2) 텍스트 프롬프트로 이미지 생성
- **목적:** 텍스트 프롬프트를 입력받아 그에 맞는 이미지를 생성합니다.
- **관찰 포인트**
    - prompt는 텍스트 설명으로, 해당 설명에 맞는 이미지를 생성
    - `guidance_scale`은 모델이 텍스트 프롬프트를 얼마나 "강력하게" 반영할지를 결정하는 매개변수임

### [셀 # 4] 코드 설명

**한 줄 요약:** 프롬프트 한 줄을 넣어 첫 번째 그림을 만들어 봅니다.

**코드 흐름**

1. `guidance_scale = 7.5` — 프롬프트를 얼마나 강하게 따를지 정하는 값입니다. 7.5는 가장 널리 쓰이는 기본값입니다.
2. `prompt = "A futuristic cityscape at sunset with flying cars"` — 그리고 싶은 내용을 영어 문장으로 적습니다("날아다니는 자동차가 있는 해 질 녘 미래 도시 풍경").
3. `image = pipe(prompt, guidance_scale=guidance_scale).images[0]`
   - `pipe(...)` : 파이프라인을 실행합니다. 진행 막대가 나타나며 기본 50단계를 거칩니다.
   - `.images` : 결과에 담긴 **이미지 리스트**입니다(한 번에 여러 장도 만들 수 있기 때문).
   - `[0]` : 그중 첫 번째 이미지
4. `image` — 셀의 **마지막 줄에 변수 이름만** 적으면 주피터가 그 내용을 화면에 표시합니다. 이미지이므로 그림이 나타납니다.

**Python 초보 포인트**

- `guidance_scale=guidance_scale` 에서 왼쪽은 **함수가 받는 옵션의 이름**, 오른쪽은 **내가 만든 변수**입니다. 이름이 같을 뿐 서로 다른 것입니다.
- 문자열은 큰따옴표나 작은따옴표로 감쌉니다.
- 같은 셀을 다시 실행하면 **매번 다른 그림**이 나옵니다. 출발점이 되는 잡음이 무작위이기 때문입니다(고정하는 방법은 6, 8번 셀).

**알아둘 점:** 코드 주석에 "창의성(높을수록 창의적, 낮을수록 구체적)"이라고 적혀 있지만, 실제 동작은 **반대**에 가깝습니다. `guidance_scale`이 **높을수록 프롬프트에 충실하게**(대신 색이 과해지고 부자연스러워질 수 있음), **낮을수록 프롬프트에서 자유롭게**(모델 마음대로) 그립니다. 6, 7번 셀에서 직접 확인할 수 있습니다.

In [ ]:
# 4
# 창의성(높을수록 창의적, 낮을수록 구체적)
guidance_scale = 7.5

# 생성할 이미지에 대한 텍스트 프롬프트 정의
prompt = "A futuristic cityscape at sunset with flying cars"

# 이미지 생성
image = pipe(prompt, guidance_scale=guidance_scale).images[0]
image

### [셀 # 5] 코드 설명

**한 줄 요약:** 프롬프트만 바꿔서("말을 탄 우주비행사, 실사풍") 그림을 한 장 더 만듭니다.

**코드 흐름**

1. `prompt = "An astronaut riding a horse in a photorealistic style"` — `prompt` 변수에 새 문장을 넣습니다. 4번 셀의 문장은 덮어써집니다.
2. `image = pipe(prompt, guidance_scale=guidance_scale).images[0]` — 4번 셀과 똑같은 줄입니다. `guidance_scale`은 4번 셀에서 정한 7.5가 그대로 쓰입니다.
3. `image` — 결과를 화면에 표시합니다.

**Python 초보 포인트**

- 변수에 새 값을 넣으면 이전 값은 사라집니다. `prompt`와 `image` 모두 새것으로 바뀝니다.
- 노트북의 변수는 **셀 사이에 공유**됩니다. 그래서 이 셀에 `guidance_scale`을 정하는 줄이 없어도 4번 셀의 값이 쓰입니다. 반대로 4번 셀을 실행하지 않고 이 셀을 실행하면 `NameError`가 납니다.

**직접 해 보기:** 프롬프트 뒤에 `", oil painting"`, `", at night"` 같은 말을 덧붙여 결과가 어떻게 달라지는지 확인해 보세요. 프롬프트는 영어로 써야 잘 동작합니다.

In [ ]:
# 5
prompt = "An astronaut riding a horse in a photorealistic style"
image = pipe(prompt, guidance_scale=guidance_scale).images[0]
image

---
## 3) 창의성 변경
- **목적:** guidance_scale을 변경하면서 결과 이미지를 확인합니다.
- **관찰 포인트**
    - guidance_scale은 모델이 프롬프트(텍스트)를 얼마나 강하게 따를 것인지를 결정하는 중요한 파라미터임.
    - 값이 낮으면 프롬프트에 매우 엄격한 이미지가 나오고, 값이 높으면 자유롭고 창의적인(하지만 엉뚱한) 이미지가 나옴.

### [셀 # 6] 코드 설명

**한 줄 요약:** 같은 프롬프트, 같은 출발 잡음(seed)에서 `guidance_scale`만 1.5 / 10.0 / 20.0으로 바꿔 그림 3장을 만들고 나란히 비교합니다. 이후 실험 셀들의 **기본 틀**이 되는 셀입니다.

**코드 흐름**

1. 설정
   - `prompt = "..."` : 프롬프트
   - `scales = [1.5, 10.0, 20.0]` : 비교할 값 3개를 리스트로 준비
   - `images = []` : 결과를 담을 빈 리스트
2. 생성 반복 — `for scale in scales:`
   - `generator = torch.manual_seed(50)` : 무작위 수의 출발점을 50번으로 **고정**합니다. 반복할 때마다 다시 50으로 맞추므로 세 그림이 **같은 잡음에서 출발**합니다. 이렇게 해야 차이가 오직 `guidance_scale` 때문이라고 말할 수 있습니다.
   - `result = pipe(prompt, guidance_scale=scale, generator=generator).images[0]` : 그림 생성
   - `images.append((scale, result))` : `(값, 그림)` 쌍을 리스트에 추가
3. 그리기
   - `plt.figure(figsize=(20, 7))` : 넓은 도화지
   - `for i, (scale, img) in enumerate(images):` : 순번과 `(값, 그림)`을 함께 꺼냅니다.
   - `plt.subplot(1, 3, i + 1)` : 1줄×3칸 중 `i+1`번째 칸(칸 번호는 1부터)
   - `plt.imshow(img)`, `plt.title(...)`, `plt.axis("off")` : 그림, 제목, 눈금 숨기기
   - `plt.tight_layout()`, `plt.show()` : 간격 정리 후 표시

**보는 법**

- **1.5 (낮음):** 프롬프트를 약하게 따릅니다. 흐릿하거나 내용이 프롬프트와 동떨어질 수 있습니다.
- **10.0 (보통):** 프롬프트 내용이 선명하게 반영됩니다.
- **20.0 (높음):** 프롬프트를 매우 강하게 따릅니다. 색과 대비가 과해지거나 부자연스러워질 수 있습니다.

**Python 초보 포인트**

- `images.append((scale, result))` 에 괄호가 두 겹인 이유: 안쪽 괄호가 튜플 `(scale, result)` 하나를 만들고, 그것을 리스트에 추가합니다.
- `enumerate(목록)` : "몇 번째인지"와 "내용"을 함께 꺼냅니다. 번호는 0부터라서 `i + 1`을 합니다.
- `for i, (scale, img) in ...` : 꺼내면서 튜플 안까지 한 번에 풀어 담습니다.
- `f"Guidance Scale: {scale}"` : f-string으로 제목에 값을 넣습니다.

In [ ]:
# 6
import torch
import matplotlib.pyplot as plt

# 1. 설정
prompt = "A high-quality fantasy landscape, a mystical river flowing through a neon forest"
scales = [1.5, 10.0, 20.0]  # 비교할 세 가지 값 (낮음, 보통, 높음)
images = []

# 2. 반복문을 통해 이미지 생성
print(f"이미지 생성 시작: {prompt}")
for scale in scales:
    print(f"현재 작업 중: guidance_scale = {scale}")
    # 이미지 생성 (추론 속도를 위해 도중에 진행 상황 표시 생략 가능)
    generator = torch.manual_seed(50) # 동일한 조건 비교를 위해 시드 고정
    result = pipe(prompt, guidance_scale=scale, generator=generator).images[0]
    images.append((scale, result))

# 3. 결과 시각화 (Matplotlib 사용)
plt.figure(figsize=(20, 7)) # 전체 창 크기 설정

for i, (scale, img) in enumerate(images):
    plt.subplot(1, 3, i + 1) # 1행 3열 중 i+1번째
    plt.imshow(img)
    plt.title(f"Guidance Scale: {scale}", fontsize=15)
    plt.axis("off") # 축 정보 숨기기

plt.tight_layout()
plt.show()

---
## 3-1) 과도한 창의성
- guidance_scale을 **매우 낮게(1.0)**부터 **매우 높게(30.0)**까지 설정해 봄
- 수치가 너무 높을 때 이미지가 어떻게 깨지는지(Over-baked) 직접 관찰해 봄

### [셀 # 7] 코드 설명

**한 줄 요약:** 6번 셀과 같은 실험을 더 극단적인 값(1.0 / 7.5 / 30.0)으로 해서, `guidance_scale`이 지나치게 높으면 그림이 어떻게 망가지는지 확인합니다.

**코드 흐름**

1. `prompt`(선글라스를 쓴 골든 리트리버의 실사 초상), `cfg_values = [1.0, 7.5, 30.0]`, `images = []` 준비
2. `for cfg in cfg_values:` — 값마다
   - `generator = torch.manual_seed(500)` : seed를 500으로 고정
   - `img = pipe(prompt, guidance_scale=cfg, generator=generator).images[0]`
   - `images.append((cfg, img))`
3. `plt.figure(figsize=(18, 6))` 후 반복문으로 3칸에 그림

**보는 법**

- **1.0:** 프롬프트의 안내가 사실상 꺼진 상태입니다. 내용이 엉뚱하거나 흐릿하게 나오기 쉽습니다.
- **7.5:** 권장값. 균형이 좋습니다.
- **30.0:** 색이 타 버린 듯 과하게 진해지고, 윤곽이 거칠어지는 현상이 나타납니다.

**Python 초보 포인트**

- `plt.imshow(img); plt.title(f"..."); plt.axis("off")` — 세미콜론(`;`)으로 **여러 문장을 한 줄에** 이어 쓸 수 있습니다. 6번 셀에서 세 줄로 쓴 것과 완전히 같은 동작입니다. 짧게 쓸 수는 있지만 읽기는 조금 불편해서 자주 쓰는 방식은 아닙니다.
- `cfg`는 Classifier-Free Guidance의 줄임말로, `guidance_scale`을 가리키는 흔한 별칭입니다.
- 변수 이름이 6번 셀(`scales`, `scale`)과 다르지만(`cfg_values`, `cfg`) 하는 일은 같습니다. 변수 이름은 자유롭게 정할 수 있습니다.

In [ ]:
# 7
prompt = "A hyper-realistic portrait of a golden retriever wearing sunglasses"
cfg_values = [1.0, 7.5, 30.0] # 1: 자유로움, 7.5: 권장, 30: 과함
images = []

for cfg in cfg_values:
    generator = torch.manual_seed(500)
    img = pipe(prompt, guidance_scale=cfg, generator=generator).images[0]
    images.append((cfg, img))

plt.figure(figsize=(18, 6))
for i, (val, img) in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img); plt.title(f"Guidance Scale: {val}"); plt.axis("off")
plt.show()

---
## 4) 시드 변경
- **목적:** Seed를 변경하면서 결과 이미지를 확인합니다.
- **관찰 포인트**
    - 프롬프트가 같아도 '랜덤 노이즈(Seed)'가 바뀌면 완전히 다른 구도의 이미지가 생성됨
    - 어떤 시드 번호가 만족스러운 결과를 내는지 찾는 과정을 체험함

### [셀 # 8] 코드 설명

**한 줄 요약:** 프롬프트는 그대로 두고 **seed(출발 잡음의 번호)만** 123 / 456 / 789로 바꿔 그림 3장을 만들어, 같은 글에서도 전혀 다른 그림이 나올 수 있음을 확인합니다.

**seed란?** 컴퓨터의 "무작위"는 사실 정해진 규칙으로 만들어지며, seed는 그 출발점 번호입니다. **같은 seed면 항상 같은 무작위 값**이 나오고, 따라서 (프롬프트와 다른 옵션이 같다면) 같은 그림이 나옵니다. 마음에 드는 그림의 seed를 적어 두면 나중에 다시 만들 수 있습니다.

**코드 흐름**

1. `prompt`(왕관을 쓴 위엄 있는 고양이, 유화풍), `seeds = [123, 456, 789]`, `images = []` 준비
2. `for seed in seeds:` — seed마다
   - `generator = torch.Generator("cuda").manual_seed(seed)` : GPU용 무작위 수 생성기를 새로 만들고 seed를 지정합니다.
   - `image = pipe(prompt, generator=generator).images[0]` : `guidance_scale`을 적지 않았으므로 기본값(7.5)이 쓰입니다.
   - `images.append((seed, image))`
3. 1줄×3칸으로 그리고 제목에 seed 번호를 표시합니다.

**보는 법:** 세 그림 모두 "왕관 쓴 고양이 유화"이지만 구도, 고양이의 생김새, 배경이 모두 다릅니다. 프롬프트는 "무엇을" 그릴지 정하고, seed는 "어떤 모습으로" 그릴지 정한다고 이해하면 됩니다.

**Python 초보 포인트**

- 프롬프트가 큰따옴표로 감싸져 있어서, 안에 작은따옴표(`king's`)가 들어 있어도 문제가 없습니다. 반대로 작은따옴표로 감쌌다면 여기서 문자열이 끊겨 에러가 납니다.
- `torch.Generator("cuda").manual_seed(seed)` 는 "생성기를 만들고 → 바로 seed를 설정"하는 메서드 체이닝입니다.
- 6, 7번 셀의 `torch.manual_seed(50)`과 비교: 그쪽은 PyTorch 전체의 기본 생성기(CPU)의 seed를 정하는 방식이고, 이쪽은 이 작업 전용 생성기를 GPU에 따로 만드는 방식입니다. 둘 다 결과를 고정하는 목적은 같습니다. 다만 같은 번호라도 **두 방식이 만드는 그림은 서로 다릅니다.**

In [ ]:
# 8
import torch
import matplotlib.pyplot as plt

prompt = "A majestic cat wearing a king's crown, oil painting style"
seeds = [123, 456, 789] # 무작위 시드 번호
images = []

for seed in seeds:
    # 동일한 시드 설정을 위해 generator 생성
    generator = torch.Generator("cuda").manual_seed(seed)
    image = pipe(prompt, generator=generator).images[0]
    images.append((seed, image))

# 결과 출력
plt.figure(figsize=(15, 5))
for i, (seed, img) in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(f"Seed: {seed}")
    plt.axis("off")
plt.show()

---
## 5) Negative Prompt 반영
- **목적:** 결과 이미지에서 원하지 않는 요소들을 제거해 봅니다.
- **관찰 포인트**
    - "이것만은 그리지 마"라고 지시하는 기법.
    - 선호하는 이미지를 얻는 효과적인 방법 중 하나 임.

### [셀 # 9] 코드 설명

**한 줄 요약:** 같은 프롬프트, 같은 seed에서 **negative prompt**(나오지 않았으면 하는 것)를 넣었을 때와 안 넣었을 때를 비교합니다.

**코드 흐름**

1. `prompt = "... a green forest with many bright red flowers, a woman standing in the center ..."` — "빨간 꽃이 가득한 초록 숲 한가운데 여성이 서 있는 수채화". (바로 위의 `# prompt = ...` 줄은 주석 처리된 이전 버전입니다.)
2. `negative_prompts = ["", "woman, female, person, red flowers, red color"]` — 두 가지 경우입니다.
   - `""` : 빈 문자열 → 아무것도 제한하지 않음
   - 두 번째 : 사람과 빨간색을 빼 달라는 요청
3. `generator_seed = 2024` — 두 그림의 출발점을 같게 하기 위한 seed
4. `for neg_prompt in negative_prompts:` — 경우마다
   - `generator = torch.Generator("cuda").manual_seed(generator_seed)`
   - `pipe(prompt, negative_prompt=neg_prompt, generator=generator, num_inference_steps=30, guidance_scale=7.5)` : 옵션 여러 개를 한 번에 지정합니다. 단계 수는 30으로 줄여 조금 빠르게 만듭니다.
   - `images.append((neg_prompt, image))`
5. 그리기
   - `fig, ax = plt.subplots(1, 2, figsize=(16, 8))` : 1줄×2칸 격자를 만들고, 칸들을 `ax`에 받습니다.
   - `titles = [...]` : 칸별 제목
   - `for i in range(2):` → `ax[i].imshow(images[i][1])` : i번째 칸에 i번째 결과의 그림을 그립니다.

**보는 법:** 프롬프트에는 분명히 "여성"과 "빨간 꽃"이 있는데도, 오른쪽 그림에서는 사람이 사라지고 빨간색이 줄어듭니다. negative prompt가 결과를 그 내용의 반대 방향으로 밀어내기 때문입니다. (seed나 프롬프트에 따라 완전히 사라지지 않을 수도 있습니다.) 실무에서는 `"blurry, low quality, extra fingers"` 같은 말을 넣어 품질을 높이는 데 많이 씁니다.

**Python 초보 포인트**

- `images[i][1]` : 두 단계로 꺼냅니다. `images[i]`는 `(negative prompt, 그림)` 튜플이고, 거기서 `[1]`이 그림입니다(`[0]`은 문자열).
- 지금까지는 `plt.subplot(1, 3, i+1)` 방식이었고, 여기서는 `fig, ax = plt.subplots(...)` 방식입니다. 이름이 비슷하지만(`subplot` / `subplots`) 사용법이 다릅니다. 후자는 칸들을 한꺼번에 만들어 `ax[0]`, `ax[1]`로 접근하고, 제목은 `ax[i].set_title(...)`로 붙입니다.
- 제목 문자열 안의 `\n` 은 줄바꿈입니다.
- `range(2)` 는 0, 1 입니다.

In [ ]:
# 9
import torch
import matplotlib.pyplot as plt

# 1. 프롬프트 설정 (긍정 프롬프트는 동일하게 유지)
# 빨간 꽃이 가득한 숲속에 한 여성이 서 있는 수채화
# prompt = "A beautiful watercolor painting of a lush green forest with many bright red flowers, a woman standing in the center, high quality"
prompt = "A beautiful watercolor painting of a green forest with many bright red flowers, a woman standing in the center, high quality"

# 2. 네거티브 프롬프트 설정
# 첫 번째는 아무것도 제한하지 않음, 두 번째는 '사람'과 '빨간색'을 강력하게 제한
negative_prompts = [
    "",
    "woman, female, person, red flowers, red color"
]

images = []
generator_seed = 2024 # 차이를 확실히 보기 위해 구도 고정

for neg_prompt in negative_prompts:
    generator = torch.Generator("cuda").manual_seed(generator_seed)

    # 이미지 생성
    image = pipe(
        prompt,
        negative_prompt=neg_prompt,
        generator=generator,
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]

    images.append((neg_prompt, image))

# 3. 결과 비교 시각화
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

titles = ["No Negative Prompt (Original)", "With Negative Prompt\n(No Woman, No Red)"]

for i in range(2):
    ax[i].imshow(images[i][1])
    ax[i].set_title(titles[i], fontsize=15, pad=20)
    ax[i].axis("off")

plt.tight_layout()
plt.show()

---
## 6) Inference Steps 변경
- **목적:** 결과 이미지 생성 단계를 조절하면서 결과 이미지 변화를 확인합니다.
- **관찰 포인트**
    - 그림을 얼마나 정성 들여 그릴지(단계)를 조절함
    - 단계가 높을수록 디테일해지지만 시간이 오래 걸림

### [셀 # 10] 코드 설명

**한 줄 요약:** 잡음을 걷어 내는 반복 횟수(`num_inference_steps`)를 5 / 20 / 100으로 바꿔, 단계 수가 그림 품질에 어떤 영향을 주는지 비교합니다.

**코드 흐름**

1. `prompt`(미래적인 사이버펑크 도시, 매우 세밀하게), `steps_list = [5, 20, 100]`, `images = []` 준비
2. `for steps in steps_list:` — 값마다
   - `generator = torch.manual_seed(1004)` : seed 고정
   - `image = pipe(prompt, num_inference_steps=steps, generator=generator).images[0]`
   - `images.append((steps, image))`
3. 1줄×3칸으로 그리고 제목에 단계 수를 표시합니다.

**보는 법**

- **5단계:** 잡음을 걷어 낼 기회가 너무 적어 흐릿하고 뭉개진 그림이 나옵니다.
- **20단계:** 꽤 그럴듯한 그림이 나옵니다.
- **100단계:** 조금 더 다듬어지지만, 20단계와의 차이는 5→20의 차이만큼 크지 않습니다.

단계 수와 걸리는 시간은 비례합니다(100단계는 20단계의 약 5배). 그래서 실제로는 20~50 정도에서 "품질과 시간의 타협점"을 찾습니다. 기본값은 50입니다.

**Python 초보 포인트**

- 6번 셀과 구조가 완전히 같고, 바뀐 것은 **리스트의 값**과 **`pipe`에 넘기는 옵션 이름**뿐입니다. 이처럼 틀 하나를 익히면 여러 실험에 재사용할 수 있습니다.
- 리스트 이름을 `steps`가 아니라 `steps_list`로 지은 것은, 반복문에서 꺼내는 낱개 변수 `steps`와 이름이 겹치지 않게 하기 위해서입니다.
- 이 셀은 총 125단계를 계산하므로 다른 셀보다 오래 걸립니다.

In [ ]:
# 10
prompt = "A futuristic cyberpunk city, ultra detailed"
steps_list = [5, 20, 100] # 단계 설정
images = []

for steps in steps_list:
    generator = torch.manual_seed(1004) # 비교를 위해 시드 고정
    image = pipe(prompt, num_inference_steps=steps, generator=generator).images[0]
    images.append((steps, image))

# 결과 출력
plt.figure(figsize=(15, 5))
for i, (steps, img) in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(f"Steps: {steps}")
    plt.axis("off")
plt.show()

---
## 7) Style Variation (화풍 전환)
- **목적:** 결과 이미지의 화풍을 변경시켜 봅니다.
- **관찰 포인트**
    - 동일한 대상을 두고 '스타일' 프롬프트만 바꿔서 화풍 변화를 체험함.

### [셀 # 11] 코드 설명

**한 줄 요약:** 그리는 대상(비 오는 도시 거리의 미래형 스포츠카)은 그대로 두고, 뒤에 붙이는 **화풍 설명**만 바꿔 3가지 스타일의 그림을 만듭니다.

**코드 흐름**

1. `base_prompt = "A futuristic sports car parked in a rainy city street"` — 공통으로 쓸 기본 문장
2. `styles = [...]` — 화풍 설명 3개: 사이버펑크(네온 조명), 유화(반 고흐풍 두꺼운 붓 터치), 설계도(기술 도면)
3. `for style in styles:` — 화풍마다
   - `full_prompt = f"{base_prompt}, {style}"` : 기본 문장과 화풍을 쉼표로 이어 최종 프롬프트를 만듭니다.
   - `generator = torch.manual_seed(1234)` : seed 고정
   - `img = pipe(full_prompt, generator=generator).images[0]`
   - `images.append((style.split(',')[0], img))` : 제목으로 쓸 **짧은 이름**과 그림을 함께 저장합니다.
4. 1줄×3칸으로 그립니다.

**`style.split(',')[0]` 풀이**

- `"Cyberpunk style, neon lights, 8k resolution".split(',')` → 쉼표를 기준으로 잘라 `['Cyberpunk style', ' neon lights', ' 8k resolution']` 리스트가 됩니다.
- `[0]` → 그 첫 번째인 `'Cyberpunk style'`
- 긴 화풍 설명 전체 대신 맨 앞부분만 제목으로 쓰려는 것입니다.

**보는 법:** 같은 seed로 출발했기 때문에 차의 위치나 구도가 어느 정도 비슷하게 유지되면서, 질감과 색감만 크게 달라지는 것을 볼 수 있습니다.

**Python 초보 포인트**

- `f"{a}, {b}"` : f-string으로 두 문자열을 쉼표와 함께 이어 붙입니다.
- `문자열.split(구분자)` : 문자열을 구분자로 잘라 리스트로 만듭니다.
- 리스트를 여러 줄에 걸쳐 쓸 때는 항목 끝마다 쉼표를 붙입니다.
- `for i, (title, img) in enumerate(images):` 에서 `title` 자리에 위에서 저장한 짧은 이름이 들어옵니다.

In [ ]:
# 11
# 동일한 대상, 다른 스타일
base_prompt = "A futuristic sports car parked in a rainy city street"
styles = [
    "Cyberpunk style, neon lights, 8k resolution",
    "Oil painting, thick brushstrokes, van Gogh style",
    "Blueprint, technical drawing, architectural sketch"
]

images = []
for style in styles:
    full_prompt = f"{base_prompt}, {style}"
    # 일관성을 위해 시드 고정
    generator = torch.manual_seed(1234)
    img = pipe(full_prompt, generator=generator).images[0]
    images.append((style.split(',')[0], img))

# 결과 출력 (3분할)
plt.figure(figsize=(18, 6))
for i, (title, img) in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img); plt.title(title); plt.axis("off")
plt.show()

---
## 8) 해상도(Resolution)와 구도의 관계
- **목적:** 결과 이미지의 해상도와 구도간의 상관 관계를 알아봅니다.
- **관찰 포인트**
    - Stable Diffusion v1-4는 기본 512x512 학습 모델임.
    - 가로/세로 길이를 바꿨을 때 구도가 어떻게 변하는지 확인함 (VRAM 용량이 작다면 조심해야 함)

### [셀 # 12] 코드 설명

**한 줄 요약:** 그림의 가로·세로 크기(`width`, `height`)를 정사각형 / 가로로 긴 / 세로로 긴 3가지로 바꿔, 캔버스 모양이 구도에 어떤 영향을 주는지 확인합니다.

**코드 흐름**

1. `prompt = "A full body shot of a medieval knight standing in a vast desert"` — "광활한 사막에 서 있는 중세 기사의 전신 사진"
2. `sizes = [(512, 512), (768, 512), (512, 768)]` — `(가로, 세로)` 튜플 3개의 리스트
3. `for w, h in sizes:` — 크기마다
   - `generator = torch.manual_seed(10)` : seed 고정
   - `img = pipe(prompt, width=w, height=h, generator=generator).images[0]`
   - `images.append((f"{w}x{h}", img))` : `"768x512"` 같은 문자열을 제목용으로 함께 저장
4. 1줄×3칸으로 그립니다.

**보는 법**

- **512×512:** 모델이 학습된 기본 크기. 가장 안정적입니다.
- **768×512 (가로로 긴):** 사막 풍경이 넓게 펼쳐지고 기사는 상대적으로 작아지기 쉽습니다.
- **512×768 (세로로 긴):** 서 있는 사람의 "전신"을 담기에 유리합니다.

**Python 초보 포인트**

- `for w, h in sizes:` : 튜플 `(768, 512)`를 꺼내면서 `w=768`, `h=512`로 나눠 담습니다.
- `f"{w}x{h}"` : 가운데 `x`는 그냥 글자입니다. 결과는 `"768x512"`.

**알아둘 점**

- 가로·세로 값은 **8의 배수**여야 합니다. 아니면 에러가 납니다.
- 같은 seed를 써도 **크기가 다르면 출발 잡음의 모양이 달라지므로** 세 그림은 서로 다른 그림이 됩니다. 구도만 살짝 바뀌는 것이 아닙니다.
- 512에서 많이 벗어난 큰 크기에서는 같은 대상이 두 번 그려지는 등 어색한 결과가 나오기도 하고, GPU 메모리도 더 많이 필요합니다. 메모리 부족 에러가 나면 13, 14번 셀을 참고하세요.

In [ ]:
# 12
prompt = "A full body shot of a medieval knight standing in a vast desert"
# (가로, 세로) 조합
sizes = [(512, 512), (768, 512), (512, 768)]
images = []

for w, h in sizes:
    generator = torch.manual_seed(10)
    # width와 height 파라미터 사용
    img = pipe(prompt, width=w, height=h, generator=generator).images[0]
    images.append((f"{w}x{h}", img))

plt.figure(figsize=(18, 6))
for i, (label, img) in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img); plt.title(label); plt.axis("off")
plt.show()

---
## (Optional) VRAM 체크
- 현재 내 그래픽카드의 메모리가 얼마나 남았는지 확인하는 방법
- CUDA Out of Memory (VRAM 부족) 대응 필요

### [셀 # 13] 코드 설명

**한 줄 요약:** 내 GPU의 메모리(VRAM)가 전체 얼마이고, 지금 얼마나 쓰고 있으며, 얼마나 남았는지 출력하는 함수를 만들어 실행합니다.

**코드 흐름**

1. `def check_vram():` — 인자가 없는 함수입니다.
2. `if torch.cuda.is_available():` — GPU가 있을 때만 아래를 실행하고, 없으면 `else:`의 안내문을 출력합니다.
3. `device = torch.cuda.current_device()` — 현재 사용 중인 GPU의 번호(보통 0)
4. `device_name = torch.cuda.get_device_name(device)` — GPU 이름(예: RTX 3060)
5. 메모리 값 3가지
   - `torch.cuda.get_device_properties(device).total_memory` : GPU 전체 메모리
   - `torch.cuda.memory_allocated(device)` : 지금 텐서들이 **실제로 차지하고 있는** 양
   - `torch.cuda.memory_reserved(device)` : PyTorch가 나중에 빨리 쓰려고 **미리 잡아 둔** 양(위 값을 포함)
6. `free_memory = total_memory - reserved_memory` — 여유 공간 계산
7. `print(f"... {total_memory:.2f} GB")` — 소수점 둘째 자리까지 출력
8. 마지막 줄 `check_vram()` — 함수를 실행합니다.

**`/ 1024**3` 은 무엇인가?** 위 함수들은 값을 **바이트(byte)** 단위로 돌려줍니다. 1024로 세 번 나누면(바이트 → KB → MB → GB) 기가바이트가 됩니다. `1024**3`은 1024의 세제곱입니다.

**Python 초보 포인트**

- `**` 는 거듭제곱입니다. 연산 순서상 `**`가 `/`보다 먼저 계산되므로 `a / 1024**3`은 `a / (1024**3)`과 같습니다.
- 함수 안에서 만든 변수(`device`, `total_memory` 등)는 **함수 안에서만** 존재합니다. 함수 밖에서는 쓸 수 없고, 밖에 있는 같은 이름의 변수에도 영향을 주지 않습니다.
- `{값:.2f}` : 소수점 둘째 자리까지 표시.

**알아둘 점:** 여기서 보이는 값은 **이 노트북(PyTorch)이 쓰는 양**입니다. 다른 프로그램이 쓰는 GPU 메모리는 포함되지 않으므로, 전체 상황은 터미널의 `nvidia-smi` 명령으로 확인할 수 있습니다.

In [ ]:
# 13
import torch

def check_vram():
    if torch.cuda.is_available():
        # 현재 사용 중인 장치의 인덱스
        device = torch.cuda.current_device()
        # 장치 이름 (예: Tesla T4, RTX 3060 등)
        device_name = torch.cuda.get_device_name(device)

        # 전체 VRAM
        total_memory = torch.cuda.get_device_properties(device).total_memory / 1024**3
        # 현재 할당된 VRAM
        allocated_memory = torch.cuda.memory_allocated(device) / 1024**3
        # 캐시된(예약된) VRAM
        reserved_memory = torch.cuda.memory_reserved(device) / 1024**3
        # 실제 여유 공간
        free_memory = total_memory - reserved_memory

        print(f"--- GPU 정보: {device_name} ---")
        print(f"전체 VRAM: {total_memory:.2f} GB")
        print(f"사용 중인 VRAM: {allocated_memory:.2f} GB")
        print(f"예약된 VRAM: {reserved_memory:.2f} GB")
        print(f"여유 VRAM (여유분): {free_memory:.2f} GB")
    else:
        print("GPU를 사용할 수 없는 환경입니다.")

check_vram()

---
## (Optional) VRAM 부족시 대처
- VRAM 에러가 났을 때, 런타임을 재시작하지 않고 메모리를 비워주는 코드

### [셀 # 14] 코드 설명

**한 줄 요약:** GPU 메모리 부족 에러가 났을 때, 커널을 재시작하지 않고 쓰지 않는 메모리를 비우는 함수를 만들어 실행합니다.

**코드 흐름**

1. `import gc` — Python의 "쓰레기 수거(garbage collection)" 도구입니다.
2. `def clean_memory():`
   - `gc.collect()` : 더 이상 아무도 사용하지 않는 Python 객체를 찾아 메모리에서 치웁니다.
   - `torch.cuda.empty_cache()` : PyTorch가 미리 잡아 두기만 하고 쓰지 않는 GPU 메모리(13번 셀의 "예약된" 부분)를 반납합니다.
   - 완료 메시지를 출력합니다.
3. `clean_memory()` — 함수를 실행합니다.

**중요한 한계:** 이 함수는 **지금 쓰이고 있는 것은 지우지 못합니다.** `pipe` 변수가 모델을 붙잡고 있는 한 모델이 차지한 메모리는 그대로입니다. 모델까지 내리려면 주석으로 적혀 있는 것처럼 먼저 `del pipe`를 실행한 뒤 `clean_memory()`를 불러야 합니다. 물론 그 뒤에 그림을 다시 만들려면 3번 셀부터 다시 실행해야 합니다.

**Python 초보 포인트**

- `del 변수이름` 은 그 변수를 지웁니다. 지운 뒤 그 이름을 쓰면 `NameError`가 납니다.
- `# del pipe` 는 주석 처리된 코드입니다. 필요할 때만 `#`를 지워 사용합니다.
- 이 함수 안에서 `torch`를 쓸 수 있는 것은 앞 셀들에서 이미 `import torch`를 했기 때문입니다.

**메모리 부족(CUDA out of memory)이 날 때 순서대로 해 볼 것**

1. 이 셀을 실행해 캐시를 비운다.
2. 12번 셀의 이미지 크기를 줄인다(예: 768 → 512).
3. 그래도 안 되면 커널을 재시작하고 1번, 3번 셀부터 다시 실행한다.

In [ ]:
# 14
import gc

def clean_memory():
    # 1. 파이썬 객체 참조 해제
    gc.collect()
    # 2. PyTorch 캐시 비우기
    torch.cuda.empty_cache()
    print("VRAM 캐시가 정리되었습니다.")

# 사용 예시
# del pipe  # 만약 모델 자체를 지우고 싶다면 객체 삭제 후 실행
clean_memory()